# PySpark Demo
This notebook demonstrates PySpark actions and transformations using a public dataset.
It highlights:
- actions vs transformations
- narrow transformations
- wide transformations
- shuffle cost using `explain()` and timed actions

The dataset is the New York Times COVID-19 state-level dataset from GitHub.


The demo uses a public dataset with state-level COVID cases and deaths.

- `filter()`, `select()`, `withColumn()` are narrow transformations.
- `groupBy()`, `agg()`, `join()` are wide transformations that trigger a shuffle.
- `count()` and `show()` are actions that materialize the computation.


In [1]:
!java --version

openjdk 21.0.10 2026-01-20 LTS
OpenJDK Runtime Environment Zulu21.48+17-CA (build 21.0.10+7-LTS)
OpenJDK 64-Bit Server VM Zulu21.48+17-CA (build 21.0.10+7-LTS, mixed mode, sharing)


In [2]:
import os
import urllib.request
import time

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, to_date
import pyspark.pandas as ps
import pandas as pd

spark = SparkSession.builder \
    .appName("PySpark Action vs Transformation Demo") \
    .master("local[*]") \
    .getOrCreate()

spark.conf.set("spark.sql.shuffle.partitions", spark.sparkContext.defaultParallelism)


/Users/sctp/miniconda3/envs/pyspark/lib/python3.11/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/07 22:11:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark

In [5]:
# Set ANSI compliance to false so we can use spark.pandas API
spark.conf.set("spark.sql.ansi.enabled", "false")

In [6]:

url = "https://raw.githubusercontent.com/nytimes/covid-19-data/master/us-states.csv"
data_path = "./data/us_states_covid.csv"
if not os.path.exists(data_path):
    print(f"Downloading dataset to {data_path}...")
    urllib.request.urlretrieve(url, data_path)


In [7]:
covid_df = spark.read.option("header", "true").option("inferSchema", "true").csv(data_path)
covid_df = covid_df.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

print("Dataset schema")
covid_df.printSchema()

print("Sample rows")
covid_df.show(5, truncate=False)

print("Row count")
print(covid_df.count())


Dataset schema
root
 |-- date: date (nullable = true)
 |-- state: string (nullable = true)
 |-- fips: integer (nullable = true)
 |-- cases: integer (nullable = true)
 |-- deaths: integer (nullable = true)

Sample rows
+----------+----------+----+-----+------+
|date      |state     |fips|cases|deaths|
+----------+----------+----+-----+------+
|2020-01-21|Washington|53  |1    |0     |
|2020-01-22|Washington|53  |1    |0     |
|2020-01-23|Washington|53  |1    |0     |
|2020-01-24|Illinois  |17  |1    |0     |
|2020-01-24|Washington|53  |1    |0     |
+----------+----------+----+-----+------+
only showing top 5 rows
Row count
61942


In [8]:
# Narrow transformations
#
# Narrow transformations operate on each partition independently and do not require data
# to move between partitions.
# Examples: filter(), select(), withColumn().


In [9]:
narrow_df = covid_df.filter(col("state") == "New York").select("date", "state", "cases", "deaths")

print("Narrow transformation plan")
narrow_df.explain(True)


Narrow transformation plan
== Parsed Logical Plan ==
'Project ['date, 'state, 'cases, 'deaths]
+- Filter (state#18 = New York)
   +- Project [to_date(date#17, Some(yyyy-MM-dd), Some(Asia/Singapore), false) AS date#23, state#18, fips#19, cases#20, deaths#21]
      +- Relation [date#17,state#18,fips#19,cases#20,deaths#21] csv

== Analyzed Logical Plan ==
date: date, state: string, cases: int, deaths: int
Project [date#23, state#18, cases#20, deaths#21]
+- Filter (state#18 = New York)
   +- Project [to_date(date#17, Some(yyyy-MM-dd), Some(Asia/Singapore), false) AS date#23, state#18, fips#19, cases#20, deaths#21]
      +- Relation [date#17,state#18,fips#19,cases#20,deaths#21] csv

== Optimized Logical Plan ==
Project [cast(gettimestamp(date#17, yyyy-MM-dd, TimestampType, try_to_date, Some(Asia/Singapore), false) as date) AS date#23, state#18, cases#20, deaths#21]
+- Filter (isnotnull(state#18) AND (state#18 = New York))
   +- Relation [date#17,state#18,fips#19,cases#20,deaths#21] csv

== 

In [10]:
print("Count of New York rows")
print(narrow_df.count())


Count of New York rows
1118


In [11]:
sample_df = narrow_df.orderBy("date").limit(5)
print("Sample of narrow result")
sample_df.show(truncate=False)


Sample of narrow result
+----------+--------+-----+------+
|date      |state   |cases|deaths|
+----------+--------+-----+------+
|2020-03-01|New York|1    |0     |
|2020-03-02|New York|1    |0     |
|2020-03-03|New York|2    |0     |
|2020-03-04|New York|11   |0     |
|2020-03-05|New York|22   |0     |
+----------+--------+-----+------+



In [12]:
# Wide transformations
#
# Wide transformations require data to be shuffled between partitions.
# Examples: groupBy(), agg(), join().


In [13]:
wide_df = covid_df.groupBy("state").agg(
    spark_sum("cases").alias("total_cases"),
    spark_sum("deaths").alias("total_deaths")
)


In [14]:
print("Wide transformation plan")
wide_df.explain(True)


Wide transformation plan
== Parsed Logical Plan ==
'Aggregate ['state], ['state, 'sum('cases) AS total_cases#80, 'sum('deaths) AS total_deaths#81]
+- Project [to_date(date#17, Some(yyyy-MM-dd), Some(Asia/Singapore), false) AS date#23, state#18, fips#19, cases#20, deaths#21]
   +- Relation [date#17,state#18,fips#19,cases#20,deaths#21] csv

== Analyzed Logical Plan ==
state: string, total_cases: bigint, total_deaths: bigint
Aggregate [state#18], [state#18, sum(cases#20) AS total_cases#80L, sum(deaths#21) AS total_deaths#81L]
+- Project [to_date(date#17, Some(yyyy-MM-dd), Some(Asia/Singapore), false) AS date#23, state#18, fips#19, cases#20, deaths#21]
   +- Relation [date#17,state#18,fips#19,cases#20,deaths#21] csv

== Optimized Logical Plan ==
Aggregate [state#18], [state#18, sum(cases#20) AS total_cases#80L, sum(deaths#21) AS total_deaths#81L]
+- Project [state#18, cases#20, deaths#21]
   +- Relation [date#17,state#18,fips#19,cases#20,deaths#21] csv

== Physical Plan ==
AdaptiveSparkPla

In [15]:
print("Number of result rows")
print(wide_df.count())


Number of result rows
56


In [16]:
state_meta = spark.createDataFrame(
    [
        ("New York", "Northeast"),
        ("California", "West"),
        ("Texas", "South"),
        ("Florida", "South"),
        ("Illinois", "Midwest")
    ],
    ["state", "region"]
)

joined_df = wide_df.join(state_meta, on="state", how="left")
print("Joined result schema")
joined_df.printSchema()
joined_df.show(5, truncate=False)


Joined result schema
root
 |-- state: string (nullable = true)
 |-- total_cases: long (nullable = true)
 |-- total_deaths: long (nullable = true)
 |-- region: string (nullable = true)

+--------------+-----------+------------+------+
|state         |total_cases|total_deaths|region|
+--------------+-----------+------------+------+
|Utah          |626051174  |3194695     |NULL  |
|Florida       |4065043272 |52694427    |South |
|North Carolina|1775818771 |17314387    |NULL  |
|Indiana       |1153802424 |16853649    |NULL  |
|Ohio          |1813202197 |25744838    |NULL  |
+--------------+-----------+------------+------+
only showing top 5 rows


In [17]:
# Compare runtime for a narrow action and a wide action.
def time_action(df, label):
    start = time.time()
    count = df.count()
    print(f"{label}: count={count}, elapsed={time.time() - start:.3f}s")

print("\nTiming narrow transformation action")
time_action(narrow_df, "Narrow filter/select")

print("\nTiming wide transformation action")
time_action(wide_df, "Wide groupBy (shuffle)")

print("\nThe wide groupBy above triggers a shuffle. Look for 'Exchange' in the explain plan.")



Timing narrow transformation action
Narrow filter/select: count=1118, elapsed=0.078s

Timing wide transformation action
Wide groupBy (shuffle): count=56, elapsed=0.074s

The wide groupBy above triggers a shuffle. Look for 'Exchange' in the explain plan.


## Show Spark Pandas API
This section compares in-memory pandas with the pandas API on Spark.
It then uses a Spark join example to explain when shuffle happens and how projection works after the join.

In [18]:
import pandas as pd

pandas_df1 = pd.DataFrame(
    {
        "state": ["New York", "California", "Texas", "Florida", "Illinois"],
        "population_millions": [19.8, 39.5, 29.0, 22.2, 12.6],
    }
)
pandas_df2 = pd.DataFrame(
    {
        "state": ["New York", "California", "Texas", "Nevada", "Arizona"],
        "total_cases": [5000000, 10000000, 8000000, 1200000, 1500000],
    }
)

print("Pandas join result (in-memory, no Spark shuffle)")
pandas_joined = pandas_df1.merge(pandas_df2, on="state", how="left")
print(pandas_joined)


Pandas join result (in-memory, no Spark shuffle)
        state  population_millions  total_cases
0    New York                 19.8    5000000.0
1  California                 39.5   10000000.0
2       Texas                 29.0    8000000.0
3     Florida                 22.2          NaN
4    Illinois                 12.6          NaN


In [26]:
print("\nAttempting pyspark.pandas join example")

ps_df1 = ps.from_pandas(pandas_df1)
ps_df2 = ps.from_pandas(pandas_df2)
join_table = ps_df1.merge(ps_df2, on="state", how="left")
print("\npyspark.pandas join_table.head()")
print(join_table.head())


Attempting pyspark.pandas join example

pyspark.pandas join_table.head()
        state  population_millions  total_cases
0    New York                 19.8    5000000.0
1  California                 39.5   10000000.0
2       Texas                 29.0    8000000.0
3     Florida                 22.2          NaN
4    Illinois                 12.6          NaN


In [27]:
print("\nNow use Spark DataFrame join to explain shuffle and select")
print("The Spark join below is a wide transformation and usually triggers a shuffle.")
print("joined_df.head() shows the joined rows")
print(joined_df.head())



Now use Spark DataFrame join to explain shuffle and select
The Spark join below is a wide transformation and usually triggers a shuffle.
joined_df.head() shows the joined rows
Row(state='Utah', total_cases=626051174, total_deaths=3194695, region=None)


In [28]:
print("\nExplain the underlying Spark plan from pyspark.pandas")
print("This uses the Spark DataFrame produced by to_spark().")
join_table.to_spark().explain(True)


Explain the underlying Spark plan from pyspark.pandas
This uses the Spark DataFrame produced by to_spark().
== Parsed Logical Plan ==
Project [state#270, population_millions#271, total_cases#272L]
+- Project [__index_level_0__#273L, state#270, population_millions#271, total_cases#272L, monotonically_increasing_id() AS __natural_order__#275L]
   +- Project [__index_level_0__#273L, state#270, population_millions#271, total_cases#272L]
      +- Project [distributed_sequence_id#274L AS __index_level_0__#273L, state#270, population_millions#271, total_cases#272L]
         +- AttachDistributedSequence[distributed_sequence_id#274L, state#270, population_millions#271, total_cases#272L] Index: distributed_sequence_id#274L
            +- Project [state#259 AS state#270, population_millions#260 AS population_millions#271, total_cases#269L AS total_cases#272L]
               +- Project [state#259, population_millions#260, __right_total_cases#268L AS total_cases#269L]
                  +- Join Lef

/Users/sctp/miniconda3/envs/pyspark/lib/python3.11/site-packages/pyspark/pandas/utils.py:1038: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [30]:
selected_df = joined_df.select("state", "total_cases")
print("\nselected_df.head() shows the selected projection after the join")
print(selected_df.head())




selected_df.head() shows the selected projection after the join
Row(state='Utah', total_cases=626051174)


In [ ]:
print("\nExplain the Spark plan for selected_df directly")
selected_df.explain(True)



Explain the Spark plan for selected_df directly
== Parsed Logical Plan ==
'Project ['state, 'total_cases]
+- Project [state#18, total_cases#80L, total_deaths#81L, region#102]
   +- Join LeftOuter, (state#18 = state#101)
      :- Aggregate [state#18], [state#18, sum(cases#20) AS total_cases#80L, sum(deaths#21) AS total_deaths#81L]
      :  +- Project [to_date(date#17, Some(yyyy-MM-dd), Some(Asia/Singapore), false) AS date#23, state#18, fips#19, cases#20, deaths#21]
      :     +- Relation [date#17,state#18,fips#19,cases#20,deaths#21] csv
      +- LogicalRDD [state#101, region#102], false

== Analyzed Logical Plan ==
state: string, total_cases: bigint
Project [state#18, total_cases#80L]
+- Project [state#18, total_cases#80L, total_deaths#81L, region#102]
   +- Join LeftOuter, (state#18 = state#101)
      :- Aggregate [state#18], [state#18, sum(cases#20) AS total_cases#80L, sum(deaths#21) AS total_deaths#81L]
      :  +- Project [to_date(date#17, Some(yyyy-MM-dd), Some(Asia/Singapore), f

In [37]:
# Compare join patterns and caching
print("\nStrategy 1: select columns before the join")
select_before_join = wide_df.select("state", "total_cases").join(state_meta, on="state", how="left")
select_before_join.explain(True)
print("Result sample")
select_before_join.show(5, truncate=False)

print("\nStrategy 2: join first, then select")
join_then_select = wide_df.join(state_meta, on="state", how="left").select("state", "total_cases", "region")
join_then_select.explain(True)
print("Result sample")
join_then_select.show(5, truncate=False)

print("\nStrategy 3: cache the joined result and reuse it")
cached_join = wide_df.join(state_meta, on="state", how="left").cache()
print("Materializing cached join with count()")
cached_join.count()
print("Explain cached join")
cached_join.explain(True)
print("Projected from cached join")
cached_selected = cached_join.select("state", "total_cases")
cached_selected.show(5, truncate=False)



Strategy 1: select columns before the join
== Parsed Logical Plan ==
'Join UsingJoin(LeftOuter, [state])
:- Project [state#18, total_cases#80L]
:  +- Aggregate [state#18], [state#18, sum(cases#20) AS total_cases#80L, sum(deaths#21) AS total_deaths#81L]
:     +- Project [to_date(date#17, Some(yyyy-MM-dd), Some(Asia/Singapore), false) AS date#23, state#18, fips#19, cases#20, deaths#21]
:        +- Relation [date#17,state#18,fips#19,cases#20,deaths#21] csv
+- LogicalRDD [state#101, region#102], false

== Analyzed Logical Plan ==
state: string, total_cases: bigint, region: string
Project [state#18, total_cases#80L, region#102]
+- Join LeftOuter, (state#18 = state#101)
   :- Project [state#18, total_cases#80L]
   :  +- Aggregate [state#18], [state#18, sum(cases#20) AS total_cases#80L, sum(deaths#21) AS total_deaths#81L]
   :     +- Project [to_date(date#17, Some(yyyy-MM-dd), Some(Asia/Singapore), false) AS date#23, state#18, fips#19, cases#20, deaths#21]
   :        +- Relation [date#17,st